Notebook para extracción de text de un pdf con imagenes escaneadas.


In [1]:
#Verifico que terminal kernel se esta usando
import sys
print(sys.executable)

c:\Users\Angelica\Documents\Temporal-Carrera\PerceivoAI\REPOS_GITHUB\.venv\Scripts\python.exe


In [2]:
import os

In [3]:
def archivo_contenido(archivo):
    if not os.path.exists(archivo): # si no existe el archivo lo crea
        with open(archivo, 'w', encoding='utf-8') as file:
            pass

    # verifica su contenido
    with open(archivo, 'r', encoding='utf-8') as file:
        docs_procesados= list(file.read().splitlines())
    
    # print(f'Documentos procesados: {docs_procesados}')
    return docs_procesados

In [4]:
def generate_no_procesados(ruta_docs_pdf):
    procesados = archivo_contenido('procesados_pruebas.txt')
    docs_no_procesados= []
    # verificamos los archivos en carpeta de docs
    for filename in os.listdir(ruta_docs_pdf):
        # verificar si el archivo se encuentra en procesados.txt
        if filename not in procesados:
            # print('El archivo no ha sido procesado')
            ruta_completa= os.path.join(ruta_docs_pdf,filename)
            docs_no_procesados.append(ruta_completa)
        # else:
        #     print('Todos los archivos han sido procesados')

    print(f'Documentos para procesar ({len(docs_no_procesados)}):')
    for doc in docs_no_procesados:
        print(doc)
    return docs_no_procesados

In [6]:
from pdf2image import convert_from_path
from PyPDF2 import PdfReader
import pytesseract # para extraer el texto 
from pathlib import Path
import hashlib
import tqdm

pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

def extraccion_ocr_metadata(ruta_completa, filename):
  '''
  Carga, procesa por página y genera cada metadata(streaming)
  '''
  docs_metadata= []
  document_id = hashlib.md5(filename.encode()).hexdigest() # codificamos solo el nombre del archivo

  # carga el pdf
  reader = PdfReader(ruta_completa)
  total_pages = len(reader.pages)

  # convierte cada pag a una imagen
  # imagenes= convert_from_path(ruta_completa)
  # total_pages = len(imagenes)

  print(' Extracción y limpieza ')
  for nro_pag in tqdm.tqdm(range(1,total_pages+1)):
      # carga solo una página como imagen
      imagen= convert_from_path(ruta_completa, first_page= nro_pag, last_page=nro_pag, 
              poppler_path=r"C:\Users\Angelica\AppData\Roaming\Microsoft\Windows\Network Shortcuts\Release-24.07.0-0\poppler-24.07.0\Library\bin" )[0]
# "C:\Users\Angelica\AppData\Roaming\Microsoft\Windows\Network Shortcuts\Release-24.07.0-0\poppler-24.07.0\Library\bin\pdfinfo.exe" -v
      texto = clean_text( pytesseract.image_to_string(imagen)) #definir antes clean text
      
      # quiero generar un pdf con el texto legible
      full_texto=[]
      full_texto.append(texto)

      del imagen #liberar memoria
    
  print('✅ Extracción del texto (Limpieza por página)')
  return docs_metadata


Extraer el texto de un pdf 'limpio'

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

def extraccion_page(ruta):
    loader = PyPDFLoader(ruta)
    pages = loader.load()
    # async for page in loader.alazy_load():
    #     pages.append(page)
    print('✅ Extracción realizada')
    return pages

In [ ]:
import re

def clean_text(text: str) -> str:
    text = re.sub(r'©.*?\n', '', text)  # remueve símbolos de copyright y similares
    text = re.sub(r'\n+', ' ', text)  # convierte múltiples saltos de línea en espacio
    text = re.sub(r'\s{2,}', ' ', text)  # remueve espacios extra
    text = re.sub(r'\b\d{1,2}:\d{2}\b\s?', '', text) # remueve marca de tiempos
    return text.strip()

In [ ]:
import tqdm
import time
prueba = ['../doc_pdf/223221647-ECN-BusinessPath-fulldoc.pdf']

# for ruta_archivo in  tqdm.tqdm(prueba):#docs_no_procesados:
def proceso_completo(docs_no_procesados,carpeta_ouput):
    '''
    Parámetro de entrada: Lista con todas las rutas de los archivos no procesados
    '''
    if docs_no_procesados:
        for ruta_archivo in  docs_no_procesados:#prueba:
            filename = Path(ruta_archivo).name
            print(f'📌 Generando Embeddings para {filename} ...') 
            # definir una funcion para aplicar  
            time1= time.time()
            pags=extraccion_page(ruta_archivo)
            # if all(not page.page_content for page in pags):
            #     print('⚠️ PDF con imágenes detectado. Aplicando OCR ...')
            #     docs_metadata = extraccion_ocr_metadata(ruta_archivo, filename)
            # else:
            #     docs_metadata = generate_metadata(ruta_archivo, pags)
            
            # exportacion_json(docs_embedd,carpeta_ouput,filename)
            time3=time.time()
            segundos= time3-time1 
            print(f'\nTiempo total: {segundos:.2f} segundos - {segundos/60:.2f} minutos')
            print('🎉 Realizado: Generación de PDF limpio.\n\n')

    else:
        print('No hay documentos por procesar')

In [23]:
from langchain_community.document_loaders import PyPDFLoader
import re
from reportlab.lib.pagesizes import A4
from reportlab.pdfgen import canvas
from reportlab.lib.units import cm
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet


from PyPDF2 import PdfReader

# --------- Extracción ----------
def extraccion_con_saltos(ruta):
    reader = PdfReader(ruta)
    texto_paginas = []
    for page in reader.pages:
        text = page.extract_text()  # mantiene saltos de párrafo mejor que PyPDFLoader
        if text:
            texto_paginas.append(text.strip())
    return texto_paginas


# --------- Limpieza ----------
def clean_text(text: str) -> str:
    text = re.sub(r'©.*?\n', '', text)  # remueve copyright
    text = re.sub(r'\s+\n', '\n', text)  # elimina espacios antes de salto
    text = re.sub(r'\n{3,}', '\n\n', text)  # máximo 2 saltos seguidos
    text = re.sub(r' {2,}', ' ', text)  # espacios extra
    text = re.sub(r'\b\d{1,2}:\d{2}\b\s?', '', text)  # timestamps
    return text.strip()


# --------- Generar nuevo PDF ----------
def generar_pdf(texto, salida):
    doc = SimpleDocTemplate(salida, pagesize=A4)
    styles = getSampleStyleSheet()
    story = []

    # Dividir en párrafos
    for parrafo in texto.split("\n\n"):
        story.append(Paragraph(parrafo, styles["Normal"]))
        story.append(Spacer(1, 12))

    doc.build(story)
    print(f"📄 PDF generado en: {salida}")

# --------- Uso ----------
if __name__ == "__main__":
    ruta_pdf = '../docs_pdf_prueba/doc_bendezu.pdf'
    salida_pdf = '../docs_pdf_prueba/doc_bendezu_v2.pdf'

    # Extraer
    pages = extraccion_con_saltos(ruta_pdf)

    # Limpiar y unir
    # texto_total = " ".join([clean_text(p.page_content) for p in pages])

    texto_total = "\n\n".join([clean_text(p) for p in pages]) 
    # Generar nuevo PDF
    generar_pdf(texto_total, salida_pdf)


📄 PDF generado en: ../docs_pdf_prueba/doc_bendezu_v2.pdf


## Ejecucion

In [7]:
procesados_pdf = archivo_contenido('procesados_pruebas.txt')

In [12]:
ruta_docs_pdf= '../docs_pdf_prueba'
generate_no_procesados(ruta_docs_pdf)

Documentos para procesar (1):
../docs_pdf_prueba\doc_bendezu.pdf


['../docs_pdf_prueba\\doc_bendezu.pdf']

Haciendo una prueba:


In [ ]:
from pathlib import Path
ruta_prueba = '../doc_pdf/prueba.pdf'
filename= Path(ruta_prueba).name